# CardioSafe AI - Notebook 04: calibration, uncertainty, conformal prediction

Uses the predictions and weights saved by notebooks 02 and 03. Almost no training; **GPU runtime recommended** (only the MC-dropout step uses it; takes about a minute).

What we do:
1. **Calibration:** do the model's probabilities mean what they say? Temperature scaling, ECE, Brier, reliability diagrams.
2. **Uncertainty:** which uncertainty signal best flags predictions that are likely wrong? Single-model entropy vs MC dropout vs deep ensemble (entropy and disagreement), judged by risk-coverage curves.
3. **Conformal prediction:** a statistically grounded "uncertain" outcome per finding. This is what will feed the "insufficient evidence" result in the combined report.

Deployed predictor = the **deep ensemble of 15 models** (5 architectures x 3 seeds).

Caveat: the validation fold (9) is used for early stopping, threshold tuning, temperature and conformal calibration. That is acceptable for a prototype; with more data we would use a separate calibration split. The test fold (10) is only used for reporting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'
PRED = WORK + '/preds'; CKPT = WORK + '/models'
sys.path.insert(0, WORK)
print(sorted(os.listdir(PRED))[:6], '...', len(os.listdir(PRED)), 'prediction files')

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_trust.py
"""Calibration, uncertainty and conformal prediction utilities (also reused by the demo app)."""
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score

EPS = 1e-6


# ------------------------------------------------------------ calibration
def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def apply_T(p, T):
    """Temperature scaling on probabilities (equivalent to dividing logits by T)."""
    T = np.asarray(T)
    return 1 / (1 + np.exp(-logit(p) / T))


def nll(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def fit_temperature(y, p):
    """Single global temperature minimising validation BCE."""
    r = minimize_scalar(lambda t: nll(y, apply_T(p, t)), bounds=(0.05, 10), method='bounded')
    return float(r.x)


def fit_temperature_per_class(y, p):
    return np.array([fit_temperature(y[:, c:c + 1], p[:, c:c + 1]) for c in range(y.shape[1])])


def ece_binary(y, p, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    e, n = 0.0, len(p)
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.sum() / n * abs(y[m].mean() - p[m].mean())
    return float(e)


def calibration_metrics(y, p, bins=15):
    """Class-averaged ECE, Brier score and NLL."""
    ece = np.mean([ece_binary(y[:, c], p[:, c], bins) for c in range(y.shape[1])])
    return {'ECE': float(ece), 'Brier': float(((p - y) ** 2).mean()), 'NLL': nll(y, p)}


def reliability_curve(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    xs, ys, ns = [], [], []
    for b in range(bins):
        m = idx == b
        if m.sum() >= 5:
            xs.append(p[m].mean()); ys.append(y[m].mean()); ns.append(int(m.sum()))
    return np.array(xs), np.array(ys), np.array(ns)


# ------------------------------------------------------------ uncertainty
def entropy_score(p):
    """Mean binary predictive entropy over classes -> (N,)."""
    p = np.clip(p, EPS, 1 - EPS)
    return (-(p * np.log(p) + (1 - p) * np.log(1 - p))).mean(1)


def disagreement_score(P):
    """P: (members, N, C) probabilities. Mean over classes of the std across members -> (N,)."""
    return P.std(0).mean(1)


def mc_dropout_predict(model, X, passes=30, bs=512):
    """Monte-Carlo dropout: only Dropout layers stay stochastic (BatchNorm stays in eval mode)."""
    import torch
    import torch.nn as nn
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()
    out = []
    with torch.no_grad():
        for _ in range(passes):
            ps = [torch.sigmoid(model(X[i:i + bs]).float()).cpu() for i in range(0, len(X), bs)]
            out.append(torch.cat(ps).numpy())
    model.eval()
    return np.stack(out)


def risk_coverage(y, p, score, ths, coverages=(1.0, 0.9, 0.8, 0.7, 0.6, 0.5)):
    """Keep the `coverage` fraction of samples with the lowest uncertainty; report quality on what is kept."""
    order = np.argsort(score)
    rows = []
    for q in coverages:
        keep = order[:int(round(q * len(y)))]
        yk, pk = y[keep], p[keep]
        auc = np.mean([roc_auc_score(yk[:, c], pk[:, c]) for c in range(y.shape[1])])
        ham = float(((pk >= ths) != (yk == 1)).mean())
        rows.append({'coverage': q, 'AUROC': float(auc), 'error_rate': ham})
    return pd.DataFrame(rows)


# ------------------------------------------------------ conformal prediction
def conformal_calibrate(y_cal, p_cal):
    """Class-conditional (Mondrian) split conformal. Stores sorted nonconformity scores per (class, true label)."""
    cal = {}
    for c in range(y_cal.shape[1]):
        pos = y_cal[:, c] == 1
        cal[(c, 1)] = np.sort(1 - p_cal[pos, c])       # score of a positive = 1 - p
        cal[(c, 0)] = np.sort(p_cal[~pos, c])          # score of a negative = p
    return cal


def conformal_pvalues(cal, p):
    """p-value for label 1 and label 0 of each (sample, class) -> array (N, C, 2) ordered [label0, label1]."""
    N, C = p.shape
    out = np.zeros((N, C, 2))
    for c in range(C):
        for lab in (0, 1):
            s = (1 - p[:, c]) if lab == 1 else p[:, c]
            arr = cal[(c, lab)]
            ge = len(arr) - np.searchsorted(arr, s, side='left')      # count of cal scores >= s
            out[:, c, lab] = (ge + 1) / (len(arr) + 1)
    return out


def conformal_outcome(pv, alpha=0.1):
    """0 = confident negative, 1 = confident positive, 2 = uncertain (both labels plausible), 3 = empty (neither)."""
    in0, in1 = pv[..., 0] > alpha, pv[..., 1] > alpha
    out = np.full(in0.shape, 3)
    out[in0 & ~in1] = 0
    out[~in0 & in1] = 1
    out[in0 & in1] = 2
    return out


def conformal_report(y, pv, alpha=0.1):
    oc = conformal_outcome(pv, alpha)
    true_in = np.where(y == 1, pv[..., 1] > alpha, pv[..., 0] > alpha)       # true label inside the set
    confident = (oc == 0) | (oc == 1)
    err_conf = (oc[confident] == 1) != (y[confident] == 1)
    return {
        'alpha': alpha,
        'coverage_of_true_label': float(true_in.mean()),
        'frac_confident': float(confident.mean()),
        'frac_uncertain_both': float((oc == 2).mean()),
        'frac_empty': float((oc == 3).mean()),
        'error_rate_when_confident': float(err_conf.mean()) if confident.any() else float('nan'),
    }


In [ ]:
import importlib, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import cardiosafe_common as C, cardiosafe_models as MZ, cardiosafe_trust as T
importlib.reload(T)
from sklearn.metrics import roc_auc_score

X, Y, fold, tr, va, te, meta = C.load_data(WORK + '/ptbxl_100hz_super5.npz')
yv, yt = Y[va], Y[te]
def load(name):
    z = np.load(f'{PRED}/{name}.npz'); return z['val'], z['test']

ARCHS = ['resnet1d', 'xresnet1d', 'inception1d', 'bigru', 'cnn_transformer']
MEMBERS = [f'{a}_s{s}' for a in ARCHS for s in (0, 1, 2)]
PV = np.stack([load(m)[0] for m in MEMBERS]); PT = np.stack([load(m)[1] for m in MEMBERS])
ens_v, ens_t = PV.mean(0), PT.mean(0)
print('members:', PV.shape[0], '| ensemble test AUROC:', round(C.macro_auc(yt, ens_t), 4))

## 1. Calibration

In [ ]:
models = {
    'cnn1d_s0 (no aug)':      load('cnn1d_s0'),
    'resnet1d_s0':            load('resnet1d_s0'),
    'resnet1d_focal_s0':      load('resnet1d_focal_s0'),
    'xresnet1d_seedavg':      load('xresnet1d_seedavg'),
    'ENSEMBLE (15 models)':   (ens_v, ens_t),
}
rows, cal_probs = [], {}
for name, (pv, pt) in models.items():
    Tfit = T.fit_temperature(yv, pv)
    pc = T.apply_T(pt, Tfit)
    cal_probs[name] = (pv, pt, T.apply_T(pv, Tfit), pc, Tfit)
    b, a = T.calibration_metrics(yt, pt), T.calibration_metrics(yt, pc)
    rows.append({'model': name, 'T': round(Tfit, 3), 'ECE_before': b['ECE'], 'ECE_after': a['ECE'],
                 'Brier_before': b['Brier'], 'Brier_after': a['Brier'], 'NLL_before': b['NLL'], 'NLL_after': a['NLL'],
                 'AUROC_before': C.macro_auc(yt, pt), 'AUROC_after': C.macro_auc(yt, pc)})
cal_df = pd.DataFrame(rows).set_index('model')
cal_df.to_csv(WORK + '/results_04_calibration.csv')
display(cal_df.round(4))
print('T > 1 means the model was over-confident (probabilities are softened); T < 1 means under-confident.')
print('AUROC should not change: temperature scaling is monotonic.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for j, name in enumerate(['resnet1d_s0', 'resnet1d_focal_s0', 'ENSEMBLE (15 models)']):
    pv, pt, pcv, pc, Tfit = cal_probs[name]
    for i, (p, lab) in enumerate([(pt, 'before'), (pc, f'after (T={Tfit:.2f})')]):
        ax = axes[i, j]
        xs, ys, ns = T.reliability_curve(yt.ravel(), p.ravel(), bins=10)
        ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.plot(xs, ys, 'o-', color='#3b7ddd')
        ax.set_title(f'{name}\n{lab}  ECE={T.calibration_metrics(yt, p)["ECE"]:.3f}', fontsize=10)
        ax.set_xlabel('predicted probability'); ax.set_ylabel('observed frequency')
plt.tight_layout(); plt.savefig(WORK + '/fig_reliability.png', dpi=150); plt.show()

# per-class view for the ensemble
pv, pt, pcv, pc, Tfit = cal_probs['ENSEMBLE (15 models)']
print('Ensemble per-class ECE (before -> after global T):')
for c, n in enumerate(C.CLASSES):
    print(f'  {n:5s} {T.ece_binary(yt[:, c], pt[:, c]):.4f} -> {T.ece_binary(yt[:, c], pc[:, c]):.4f}')

## 2. Uncertainty: which signal flags unreliable predictions best?
For each method we score every test ECG with an uncertainty value, then **reject the most uncertain ones** and check whether quality on the remaining ECGs improves (risk-coverage). A good uncertainty signal makes quality rise as coverage falls; a useless one (random) stays flat.

Methods: single-model entropy, MC dropout (30 passes of the same ResNet), 3-seed ResNet ensemble, 15-model ensemble (entropy), 15-model ensemble (member disagreement).

In [ ]:
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mu, sd = C.standardize_fit(X[tr])
Xva, Xte = [C.to_tensor_ncl(X[m], mu, sd, dev) for m in (va, te)]
net = MZ.REGISTRY['resnet1d']().to(dev)
net.load_state_dict(torch.load(f'{CKPT}/resnet1d_s0.pt', map_location=dev))
mc_v = T.mc_dropout_predict(net, Xva, passes=30); mc_t = T.mc_dropout_predict(net, Xte, passes=30)
print('MC dropout passes:', mc_t.shape, '| mean per-sample std:', round(float(mc_t.std(0).mean()), 4))

In [ ]:
def calibrated(pv, pt):
    Tf = T.fit_temperature(yv, pv); return T.apply_T(pv, Tf), T.apply_T(pt, Tf)

r_v, r_t = load('resnet1d_s0')
mcm_v, mcm_t = mc_v.mean(0), mc_t.mean(0)
r3_v, r3_t = [np.mean([load(f'resnet1d_s{s}')[i] for s in (0, 1, 2)], 0) for i in (0, 1)]

methods = {}   # name -> (calibrated val probs, calibrated test probs, test uncertainty score)
for name, (pv, pt), score in [
    ('single_entropy',         (r_v, r_t),     None),
    ('mc_dropout_entropy',     (mcm_v, mcm_t), None),
    ('ensemble3_resnet_entropy', (r3_v, r3_t), None),
    ('ensemble15_entropy',     (ens_v, ens_t), None),
    ('ensemble15_disagreement', (ens_v, ens_t), 'disagree'),
]:
    cv, ct = calibrated(pv, pt)
    methods[name] = (cv, ct, T.disagreement_score(PT) if score == 'disagree' else T.entropy_score(ct))
rng = np.random.default_rng(0)
cv, ct = calibrated(ens_v, ens_t)
methods['random (reference)'] = (cv, ct, rng.random(len(yt)))

covs = (1.0, 0.9, 0.8, 0.7, 0.6, 0.5)
rc, detect = {}, {}
for name, (cv, ct, sc) in methods.items():
    ths = C.tune_thresholds(yv, cv)
    rc[name] = T.risk_coverage(yt, ct, sc, ths, covs)
    wrong = ((ct >= ths) != (yt == 1)).any(1)            # ECG with at least one wrong finding
    detect[name] = roc_auc_score(wrong, sc)

err = pd.DataFrame({n: d.set_index('coverage')['error_rate'] for n, d in rc.items()})
auc = pd.DataFrame({n: d.set_index('coverage')['AUROC'] for n, d in rc.items()})
print('Per-finding error rate on retained ECGs (lower is better; row = fraction of ECGs kept):')
display(err.round(4))
print('Macro AUROC on retained ECGs (higher is better):')
display(auc.round(4))
print('AUROC of the uncertainty score for detecting ECGs with >=1 wrong finding (0.5 = useless):')
display(pd.Series(detect).round(4).to_frame('detect_AUROC'))
err.to_csv(WORK + '/results_04_uncertainty_error.csv'); auc.to_csv(WORK + '/results_04_uncertainty_auroc.csv')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for n in err.columns:
    ls = ':' if 'random' in n else '-'
    ax[0].plot(err.index, err[n], ls, marker='o', label=n); ax[1].plot(auc.index, auc[n], ls, marker='o', label=n)
for a, t in zip(ax, ['Error rate on kept ECGs', 'Macro AUROC on kept ECGs']):
    a.set_xlabel('coverage (fraction of ECGs kept)'); a.set_title(t); a.invert_xaxis(); a.grid(alpha=.3)
ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(WORK + '/fig_risk_coverage.png', dpi=150); plt.show()

## 3. Conformal prediction on the calibrated ensemble
For each finding, the conformal method returns a set of plausible labels: **{absent}**, **{present}**, **{both}** (uncertain) or **{}**. At level alpha, the true label is inside the set at least (1 - alpha) of the time, for each class separately. "Both" is the principled 'we cannot decide' outcome.

In [ ]:
cv, ct = calibrated(ens_v, ens_t)
cal = T.conformal_calibrate(yv, cv)
rows = []
for alpha in (0.2, 0.1, 0.05):
    pv_t = T.conformal_pvalues(cal, ct)
    rep = T.conformal_report(yt, pv_t, alpha)
    oc = T.conformal_outcome(pv_t, alpha)
    unc = oc == 2
    rep['error_rate_when_uncertain (p>=0.5 rule)'] = float(((ct[unc] >= 0.5) != (yt[unc] == 1)).mean()) if unc.any() else float('nan')
    rep['ECGs with >=1 uncertain finding'] = float(unc.any(1).mean())
    rows.append(rep)
conf = pd.DataFrame(rows).set_index('alpha')
conf.to_csv(WORK + '/results_04_conformal.csv')
display(conf.round(4))
print('coverage_of_true_label should be >= 1 - alpha. The key check: error_rate_when_confident << error_rate_when_uncertain.')

pv_t = T.conformal_pvalues(cal, ct); oc = T.conformal_outcome(pv_t, 0.1)
pc_rows = []
for c, n in enumerate(C.CLASSES):
    r = T.conformal_report(yt[:, c:c + 1], pv_t[:, c:c + 1], 0.1)
    pc_rows.append({'class': n, 'coverage': r['coverage_of_true_label'], 'confident': r['frac_confident'], 'uncertain': r['frac_uncertain_both'],
                    'err_when_confident': r['error_rate_when_confident']})
print('Per class at alpha = 0.1:'); display(pd.DataFrame(pc_rows).set_index('class').round(4))

## 4. Save artifacts for the demo app

In [ ]:
import json
Tens = T.fit_temperature(yv, ens_v)
ths_ens = C.tune_thresholds(yv, T.apply_T(ens_v, Tens))
arrs = {f'cal_{c}_{lab}': cal[(c, lab)] for (c, lab) in cal}
np.savez(WORK + '/trust_artifacts.npz', T=Tens, thresholds=ths_ens, **arrs)
json.dump({'members': MEMBERS, 'classes': C.CLASSES, 'temperature': Tens, 'thresholds': ths_ens.tolist(), 'alpha_default': 0.1},
          open(WORK + '/trust_config.json', 'w'), indent=1)
print('saved: trust_artifacts.npz, trust_config.json | T =', round(Tens, 3))

## Next
Send me: the calibration table, the three uncertainty tables, and the conformal table. Then notebook 05: explainability (Grad-CAM / Integrated Gradients with a faithfulness test) and QTc measurement.